# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/api/python/) library, employing the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Install mlcroissant (uncomment if not installed)
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}\n\n{metadata.description}")

## 2. Data Overview
Review available record sets, fields, field and column names, and their Croissant `@id`s.

We will display all record sets, and for each, list the fields and columns, always referencing them by their `@id` as recommended.

In [ ]:
# List all record sets by @id and their fields
print('Available record sets:')
record_sets = list(dataset.record_sets.keys())
for i, rs_id in enumerate(record_sets):
    record_set = dataset.record_sets[rs_id]
    print(f"  {i+1}. @id: {rs_id} | name: {getattr(record_set, 'name', '[no name]')}")

    print('     Fields:')
    if hasattr(record_set, 'fields'):
        for field in record_set.fields:
            print(f"       - @id: {field['@id']} | name: {field.get('name', '[no name]')}")
    else:
        print('       [No fields found]')
    
    print('     Columns:')
    if hasattr(record_set, 'columns'):
        for column in record_set.columns:
            print(f"       - @id: {column['@id']} | name: {column.get('name', '[no name]')}")
    else:
        print('       [No columns found]')
    print()

## 3. Data Extraction
Here we will extract data from each record set into a pandas DataFrame for analysis.
All references use the `@id` of record sets and fields.


In [ ]:
# Get all available record set '@id's
record_set_ids = list(dataset.record_sets.keys())
dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f'Record set @id: {rs_id}')
            print(f'  Fields: {df.columns.tolist()}')
            print(f'  Number of records: {len(df)}')
    except Exception as ex:
        print(f'Could not load records for record set {rs_id}: {ex}')

# For further analysis, we'll pick the largest tabular record set detected
main_record_set_id = None
max_rows = 0
for rs_id, df in dataframes.items():
    if len(df) > max_rows:
        max_rows = len(df)
        main_record_set_id = rs_id

print(f"\nMain record set selected for EDA: {main_record_set_id} (rows: {max_rows})")
if main_record_set_id:
    print('Sample rows:')
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filter by a numeric field, normalize it, group by a categorical field, and demonstrate usage of Croissant `@id`.

In this example, we automatically select the first numeric field and first non-numeric field (e.g., for grouping). You can customize these as needed for your use case.

In [ ]:
# Identify a numeric and a categorical field by @id for the main record set
df = dataframes.get(main_record_set_id)

numeric_field_id = None
group_field_id = None

# Try to infer numeric/categorical fields
for c in df.columns:
    # Check if column is numeric
    if pd.api.types.is_numeric_dtype(df[c]):
        numeric_field_id = c
        break
# For group-by, pick first non-numeric field that's not the numeric one
for c in df.columns:
    if pd.api.types.is_string_dtype(df[c]) and c != numeric_field_id:
        group_field_id = c
        break

print(f"Numeric field selected: {numeric_field_id}")
print(f"Group field selected: {group_field_id}")

if numeric_field_id is not None:
    # Outlier filtering (for demo: threshold at 90th percentile)
    threshold = df[numeric_field_id].quantile(0.9)
    filtered_df = df[df[numeric_field_id] < threshold].copy()
    print(f"Filtered records with {numeric_field_id} < {threshold:.3f}:")
    display(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by categorical field if present
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped.head())

## 5. Visualization
Visualize the distribution of the numeric field and compare groups if applicable. Uses only Croissant `@id` field names.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field_id], kde=True, bins=15, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

if group_field_id is not None and numeric_field_id is not None:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df, palette='Set2')
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook we demonstrated how to use the [`mlcroissant`](https://mlcommons.github.io/croissant/api/python/) library to discover available record sets, fields, and extract data from a dataset specified with the Croissant schema.

Key findings and next steps:

- **Record sets and fields** were referenced throughout by their unique `@id` values for consistency and transparency.
- **Basic EDA** revealed the structure and some characteristics of the dataset. You can adapt these steps to your own research questions by selecting appropriate fields for deeper analysis.
- For advanced processing, see the [mlcroissant handbook](https://mlcommons.github.io/croissant/api/python/) for more details and utilities.